In [28]:
import torch
import torch.nn as nn

In [29]:
class Masked_attention(nn.Module):
    def __init__(self,dim_in,dim_out,dropout,context_length,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_key   = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_value = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.dim_key=dim_out 
        self.dropout=nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) 
    def forward(self,x):
        b,num_tokens,d_in=x.shape 
        key = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)
        attention_score = query @ key.transpose(1, 2) 
        attention_score=attention_score.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)         
        scaled_attention_score= torch.softmax(attention_score/torch.sqrt(torch.tensor(self.dim_key)),dim=-1)
        scaled_attention_score=self.dropout(scaled_attention_score)
        context_vector=scaled_attention_score @ value
        return context_vector
           

In [ ]:
## Masked attention wrapper
class Multi_head_attention(nn.Module):
    def __init__(self,d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList([Masked_attention(d_in, d_out, dropout, context_length, qkv_bias=False) for _ in range(num_heads)])
    def forward(self,x):
        return torch.concat([head(x) for head in self.heads],dim=-1)    

In [31]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], 
   [0.55, 0.87, 0.66], 
   [0.57, 0.85, 0.64], 
   [0.22, 0.58, 0.33], 
   [0.77, 0.25, 0.10], 
   [0.05, 0.80, 0.55]] 
)  
batch = torch.stack((inputs, inputs), dim=0)

In [32]:
d_in=3
d_out=2
context_length = batch.shape[1]

In [33]:
mha=Multi_head_attention(d_in, d_out, context_length, 0.0, num_heads=2)
out=mha(batch)